In [19]:
!pip install huggingface pandas scikit-learn tqdm huggingface_hub

ERROR: Could not find a version that satisfies the requirement p7zip-full (from versions: none)
ERROR: No matching distribution found for p7zip-full


In [ ]:
import sys

!conda install -c conda-forge p7zip -y

In [4]:
from huggingface_hub import notebook_login
notebook_login()


In [8]:
from huggingface_hub import HfApi

repo_id = "ControlNet/AV-Deepfake1M-PlusPlus"
api = HfApi()

info = api.dataset_info(repo_id=repo_id, files_metadata=True)
files = info.siblings
total_size = 0

for f in files:
    size_gb = (f.size or 0) / (1024 ** 3)
    total_size += size_gb
    print(f"{f.rfilename:80s} {size_gb:8.2f} GB")

print(f"Total Size: {total_size:8.2f} GB")

.gitattributes                                                                       0.00 GB
README.md                                                                            0.00 GB
testA/testA.zip.001                                                                  0.98 GB
testA/testA.zip.002                                                                  0.98 GB
testA/testA.zip.003                                                                  0.98 GB
testA/testA.zip.004                                                                  0.98 GB
testA/testA.zip.005                                                                  0.98 GB
testA/testA.zip.006                                                                  0.98 GB
testA/testA.zip.007                                                                  0.98 GB
testA/testA.zip.008                                                                  0.98 GB
testA/testA.zip.009                                                   

In [5]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()

DATA_ROOT = PROJECT_ROOT / "data" / "avdeepfake1mpp"
RAW_DIR = DATA_ROOT / "raw"
EXTRACTED_DIR = DATA_ROOT / "extracted"
METADATA_DIR = DATA_ROOT / "metadata"
MANIFEST_DIR = DATA_ROOT / "manifests"
SUBSET_DIR = DATA_ROOT / "subset"

for d in [RAW_DIR, EXTRACTED_DIR, METADATA_DIR, MANIFEST_DIR, SUBSET_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)

Project root: /home/jovyan/MSC_PROJECT
Data root: /home/jovyan/MSC_PROJECT/data/avdeepfake1mpp


In [23]:
import shutil

total, used, free = shutil.disk_usage(PROJECT_ROOT)

print(f"Total: {total / 1024**3:.2f} GB")
print(f"Used:  {used / 1024**3:.2f} GB")
print(f"Free:  {free / 1024**3:.2f} GB")

Total: 196.68 GB
Used:  116.64 GB
Free:  80.02 GB


In [27]:
metadata_files = [
    "train_metadata.json",
    "val_metadata.json",
    "testA_files.txt",
    "testB_files.txt",
]

for filename in metadata_files:
    try:
        path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            repo_type="dataset",
            local_dir=METADATA_DIR,
        )
        print("Downloaded:", path)
    except Exception as e:
        print(f"Could not download {filename}: {e}")

Downloaded: /home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/metadata/train_metadata.json
Downloaded: /home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/metadata/val_metadata.json
Downloaded: /home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/metadata/testA_files.txt
Downloaded: /home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/metadata/testB_files.txt


In [28]:
val_meta_path = METADATA_DIR / "val_metadata.json"

try:
    val_df = pd.read_json(val_meta_path, lines=True)
except ValueError:
    val_df = pd.read_json(val_meta_path)

print(val_df.shape)
print(val_df.columns.tolist())
display(val_df.head())

(77326, 11)
['file', 'original', 'split', 'modify_type', 'audio_model', 'fake_segments', 'audio_fake_segments', 'visual_fake_segments', 'video_frames', 'audio_frames', 'video_model']


,file,original,split,modify_type,audio_model,fake_segments,audio_fake_segments,visual_fake_segments,video_frames,audio_frames,video_model
0,vox_celeb_2/id01358/_1nATum8x78/00030/fake_vid...,vox_celeb_2/id01358/_1nATum8x78/00030/real.mp4,val,both_modified,yourtts,"[[4.4, 4.64]]","[[4.4, 4.64]]","[[4.4, 4.64]]",161,102656,Talklip
1,vox_celeb_2/id01358/_1nATum8x78/00027/fake_vid...,vox_celeb_2/id01358/_1nATum8x78/00027/real.mp4,val,both_modified,yourtts,"[[0.92, 1.04]]","[[0.92, 1.04]]","[[0.92, 1.04]]",166,106880,Talklip
2,vox_celeb_2/id01358/_1nATum8x78/00025/real.mp4,VoxCeleb2/dev/mp4/id01358/_1nATum8x78/00025.mp4,val,real,None,[],[],[],139,89088,None
3,vox_celeb_2/id01358/atpminqC_AU/00046/fake_vid...,vox_celeb_2/id01358/atpminqC_AU/00046/real.mp4,val,both_modified,yourtts,"[[2.74, 2.92]]","[[2.74, 2.92]]","[[2.74, 2.92]]",154,99008,Talklip
4,vox_celeb_2/id01358/atpminqC_AU/00041/real.mp4,VoxCeleb2/dev/mp4/id01358/atpminqC_AU/00041.mp4,val,real,None,[],[],[],178,114688,None


In [29]:
for col in val_df.columns:
    print("\n" + "=" * 80)
    print(col)
    print(val_df[col].value_counts(dropna=False).head(20))


file
file
silent_videos/subject_25_9yhsja3clq_vid_1_6/real.mp4               1
vox_celeb_2/id01358/_1nATum8x78/00030/fake_video_fake_audio.mp4    1
vox_celeb_2/id01358/_1nATum8x78/00027/fake_video_fake_audio.mp4    1
vox_celeb_2/id01358/_1nATum8x78/00025/real.mp4                     1
vox_celeb_2/id01358/atpminqC_AU/00046/fake_video_fake_audio.mp4    1
vox_celeb_2/id01358/atpminqC_AU/00041/real.mp4                     1
vox_celeb_2/id01358/atpminqC_AU/00040/fake_video_real_audio.mp4    1
vox_celeb_2/id01358/atpminqC_AU/00049/fake_video_real_audio.mp4    1
vox_celeb_2/id01358/y0DqbqHdcko/00073/real.mp4                     1
vox_celeb_2/id01358/bc1_8aKIQ3c/00057/fake_video_fake_audio.mp4    1
vox_celeb_2/id01358/hqGQLN6Kle0/00059/real.mp4                     1
vox_celeb_2/id01358/hqGQLN6Kle0/00059/real_video_fake_audio.mp4    1
vox_celeb_2/id01358/WpG_yFhiRLc/00019/real.mp4                     1
vox_celeb_2/id01358/WpG_yFhiRLc/00019/fake_video_real_audio.mp4    1
silent_videos/subject_1

In [9]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    allow_patterns=[
        "val/**",
        "val_metadata.json"
    ],
    local_dir=RAW_DIR,
    resume_download=True,
)

print("Downloaded validation split to:", RAW_DIR)


/opt/conda/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 55 files: 100%|██████████| 55/55 [01:45<00:00,  1.91s/it]

Downloaded validation split to: /home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/raw


In [15]:
first_parts = sorted(RAW_DIR.rglob("val.zip.001"))

if not first_parts:
    first_parts = sorted(RAW_DIR.rglob("*.zip.001"))

print(first_parts)

if not first_parts:
    raise FileNotFoundError("Could not find a .zip.001 file. Check where the dataset downloaded.")

[PosixPath('/home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/raw/val/val.zip.001')]


In [22]:
first_part = first_parts[0]

VAL_EXTRACTED_DIR = EXTRACTED_DIR / "val"

cmd = [
    "7z",
    "x",
    str(first_part),
    f"-o{VAL_EXTRACTED_DIR}",
    "-y"
]

print("Running:", " ".join(cmd))

process = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(process.stdout[-3000:])
print(process.stderr[-3000:])

if process.returncode != 0:
    raise RuntimeError("Extraction failed")
else:
    print("Extraction complete:", VAL_EXTRACTED_DIR)

Running: 7z x /home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/raw/val/val.zip.001 -o/home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/extracted/val -y

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,48 CPUs x64)

Scanning the drive for archives:
1 file, 1048576000 bytes (1000 MiB)

Extracting archive: /home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/raw/val/val.zip.001
--
Path = /home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/raw/val/val.zip.001
Type = Split
Physical Size = 1048576000
Volumes = 54
Total Physical Size = 56060004153
----
Path = val.zip
Size = 56060004153
--
Path = val.zip
Type = zip
Physical Size = 56060004153
64-bit = +

Everything is Ok

Folders: 74399
Files: 154652
Size:       59622109025
Compressed: 56060004153


Extraction complete: /home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/extracted/val


In [24]:
videos = list(VAL_EXTRACTED_DIR.rglob("*.mp4"))

print("Number of videos:", len(videos))
print(videos[:5])

Number of videos: 77326
[PosixPath('/home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/extracted/val/val/vox_celeb_2/id01076/fG7pJx__eds/00046/real.mp4'), PosixPath('/home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/extracted/val/val/vox_celeb_2/id01076/lgw47Vo6k94/00058/fake_video_fake_audio.mp4'), PosixPath('/home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/extracted/val/val/vox_celeb_2/id01076/lgw47Vo6k94/00053/real.mp4'), PosixPath('/home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/extracted/val/val/vox_celeb_2/id01076/CiyljO9DuDs/00009/real_video_fake_audio.mp4'), PosixPath('/home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/extracted/val/val/vox_celeb_2/id01076/CiyljO9DuDs/00012/real_video_fake_audio.mp4')]
